## File opening

In [11]:
import xarray as xr

file = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/level_1/2020.nc"

data = xr.open_dataset(file)
print(data)

<xarray.Dataset> Size: 36GB
Dimensions:     (valid_time: 8784, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2020-01-01 ... 2020-12-31T23...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
    expver      (valid_time) <U4 141kB ...
Data variables:
    stl1        (valid_time, latitude, longitude) float32 36GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-07T16:30 GRIB to CDM+CF via cfgrib-0.9.1...


## Combining ERA5 and In-situ

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import xarray as xr


# ------------------------------------------------------------------
# Paths – surface layer, ERA5 level 1
# ------------------------------------------------------------------
INSITU_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/in_situ/combine"
)
ERA5_L1_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/level_1"
)
OUT_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Helper: load ERA5 stl1 time series at nearest grid point for one station
# ------------------------------------------------------------------
def extract_era5_for_station(lat_station, lon_station, years):
    """
    Returns:
      era5_df: DataFrame with index 'datetime' and columns
               ['ts_station', 'lat_era5', 'lon_era5'].

      'ts_station' here is ERA5 stl1 (K) at the nearest grid cell.
    """
    lon_station_360 = lon_station % 360.0

    dfs_year = []
    lat_era5_val = None
    lon_era5_val = None

    for year in sorted(set(years)):
        f1 = ERA5_L1_ROOT / f"{year}.nc"
        if not f1.exists():
            print(f"  ERA5 level-1 file missing for year {year}, skipping this year.")
            continue

        ds1 = xr.open_dataset(f1)

        # nearest grid point in ERA5
        ds1_sel = ds1.sel(latitude=lat_station, longitude=lon_station_360, method="nearest")

        lat_era = float(ds1_sel["latitude"].values)
        lon_era_raw = float(ds1_sel["longitude"].values)
        # convert 0–360 to -180–180
        lon_era = ((lon_era_raw + 180.0) % 360.0) - 180.0

        lat_era5_val = lat_era if lat_era5_val is None else lat_era5_val
        lon_era5_val = lon_era if lon_era5_val is None else lon_era5_val

        # ERA5 level-1 soil temperature (K)
        t1 = ds1_sel["stl1"].to_series()

        df_y = pd.DataFrame(
            {
                "ts_station": t1,  # ERA5 level-1, named ts_station in output
            }
        )
        df_y.index.name = "datetime"
        dfs_year.append(df_y)

        ds1.close()

    if not dfs_year:
        return None

    era5_df = pd.concat(dfs_year).sort_index()
    era5_df["lat_era5"] = lat_era5_val
    era5_df["lon_era5"] = lon_era5_val
    return era5_df


# ------------------------------------------------------------------
# Process each network and station
# ------------------------------------------------------------------
for network_dir in sorted(INSITU_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    net_id = network_dir.name
    print(f"\n=== Network: {net_id} ===")

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue

        st_id = station_dir.name
        station_files = list(station_dir.glob("*_soil_temperature_depths.csv"))
        if not station_files:
            print(f"  {net_id}/{st_id}: no in-situ CSV, skipping.")
            continue

        csv_path = station_files[0]
        print(f"  Processing {net_id} / {st_id} -> {csv_path.name}")

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"    Failed to read {csv_path}: {e}")
            continue

        needed = [
            "date", "time",
            "ts_station_k",
            "lat", "lon",
            "elev", "cc", "lc",
            "land_cover_group", "climate_group",
            "temp_class", "elevation_class",
        ]
        missing = [c for c in needed if c not in df.columns]
        if missing:
            print(f"    Missing columns {missing}, skipping station.")
            continue

        # round in-situ Kelvin values
        df["ts_station_k"] = df["ts_station_k"].round(3)

        # build datetime index
        df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
        if df["datetime"].isna().all():
            print("    All datetimes are NaT, skipping station.")
            continue

        df = df.dropna(subset=["datetime"]).sort_values("datetime")
        df = df.set_index("datetime")

        lat_station = float(df["lat"].iloc[0])
        lon_station = float(df["lon"].iloc[0])
        years = df.index.year.unique()

        era5_df = extract_era5_for_station(lat_station, lon_station, years)
        if era5_df is None:
            print("    No ERA5 data found for this station, skipping.")
            continue

        # drop any existing in-situ ts_station (°C weighted mean) to avoid naming clash
        if "ts_station" in df.columns:
            df = df.drop(columns=["ts_station"])

        # join on datetime (inner: only overlapping timestamps)
        combined = df.join(era5_df, how="inner")
        if combined.empty:
            print("    No overlapping timestamps between station and ERA5, skipping.")
            continue

        # strict paired availability: if either in-situ or ERA5 is NaN, set both to NaN
        mask_bad = (
            combined["ts_station_k"].isna()
            | combined["ts_station"].isna()
        )
        combined.loc[mask_bad, ["ts_station_k", "ts_station"]] = np.nan

        combined = combined.reset_index()
        combined["date"] = combined["datetime"].dt.strftime("%Y-%m-%d")
        combined["time"] = combined["datetime"].dt.strftime("%H:%M")
        combined = combined.drop(columns=["datetime"])

        # drop all T_* depth columns
        drop_cols = [c for c in combined.columns if c.startswith("T_")]
        if drop_cols:
            combined = combined.drop(columns=drop_cols)

        # column ordering: in-situ first, then ERA5 ts_station, then meta
        meta_cols = [
            "date", "time",
            "ts_station_k",   # in-situ Kelvin
            "ts_station",     # ERA5 stl1 Kelvin
            "lat", "lon",
            "lat_era5", "lon_era5",
            "elev", "cc", "lc",
            "land_cover_group", "climate_group",
            "temp_class", "elevation_class",
        ]

        other_cols = [
            c for c in combined.columns
            if c not in meta_cols
        ]

        final_cols = meta_cols + other_cols
        final_cols = [c for c in final_cols if c in combined.columns]
        combined = combined[final_cols]

        out_station_dir = OUT_ROOT / net_id / st_id
        out_station_dir.mkdir(parents=True, exist_ok=True)

        out_csv = out_station_dir / f"{st_id}_insitu_era5_soil_temperature_level1.csv"
        combined.to_csv(out_csv, index=False, float_format="%.3f")

        print(f"    Wrote: {out_csv}")



=== Network: ARM ===
  Processing ARM / Anthony -> Anthony_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Anthony/Anthony_insitu_era5_soil_temperature_level1.csv
  Processing ARM / Ashton -> Ashton_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Ashton/Ashton_insitu_era5_soil_temperature_level1.csv
  Processing ARM / Byron -> Byron_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Byron/Byron_insitu_era5_soil_temperature_level1.csv
  Processing ARM / Lamont-CF1 -> Lamont-CF1_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Lamont-CF1/Lamont-CF1_insitu_era5_soil_temperature_level1.csv
  Processing ARM / MapleCity -> MapleCity_soil_temperature_depths.csv
    Wrote: /home/aminr/Desktop/Master_Thesis/Global_scale/surfac

## Checking the values of the combined file with raw era5 data

- checked the combined fill ts_era5 against raw data, and the values are correctly written 
- no file is missing or empty 

In [3]:
from pathlib import Path
import xarray as xr
import pandas as pd

ERA5_L1_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/level_1"
)
year = 2020
era5_file = ERA5_L1_ROOT / f"{year}.nc"

lat_station = 50.989
lon_station = 6.324

# Open dataset
ds = xr.open_dataset(era5_file)

# Convert station lon to 0–360 to match ERA5
lon_station_360 = lon_station % 360.0

# Select nearest gridpoint (note: still latitude/longitude)
ds_sel = ds.sel(
    latitude=lat_station,
    longitude=lon_station_360,
    method="nearest",
)

print("Nearest ERA5 grid point:")
print("  latitude :", float(ds_sel["latitude"].values))
print("  longitude:", float(ds_sel["longitude"].values))

# Times of interest (match valid_time)
times = [
    "2020-12-30T20:00:00",
    "2020-12-30T21:00:00",
    "2020-12-30T22:00:00",
    "2020-12-30T23:00:00",
]

# IMPORTANT: use 'valid_time' not 'time'
stl1_sel = ds_sel["stl1"].sel(valid_time=times)

# Convert to a small DataFrame
df = stl1_sel.to_series().reset_index()
df.rename(columns={"valid_time": "datetime", "stl1": "stl1_K"}, inplace=True)

print("\nRaw ERA5 stl1 values at nearest grid point:")
print(df.to_string(index=False))

ds.close()


Nearest ERA5 grid point:
  latitude : 51.0
  longitude: 6.25

Raw ERA5 stl1 values at nearest grid point:
           datetime     stl1_K
2020-12-30 20:00:00 277.062103
2020-12-30 21:00:00 276.973511
2020-12-30 22:00:00 276.677979
2020-12-30 23:00:00 276.607147


## Checking for missing/empty stations/files

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# Path to combined ERA5–in-situ surface-layer files
# ------------------------------------------------------------------
COMBINE_ROOT = Path(
    "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine"
)

# ------------------------------------------------------------------
# Containers for reports
# ------------------------------------------------------------------
missing_file = []        # (network, station)
missing_cols = []        # (network, station, [cols])
empty_overlap = []       # (network, station)

all_nan_ts_station_k = []   # (network, station)
all_nan_ts_station = []     # (network, station)
no_valid_pairs = []         # (network, station)

# Required columns in combined files
required_cols = [
    "date", "time",
    "ts_station_k",   # in-situ (K)
    "ts_station",     # ERA5 stl1 (K)
    "lat", "lon",
    "lat_era5", "lon_era5",
    "elev", "cc", "lc",
    "land_cover_group", "climate_group",
    "temp_class", "elevation_class",
]

# ------------------------------------------------------------------
# Scan all networks and stations under COMBINE_ROOT
# ------------------------------------------------------------------
if not COMBINE_ROOT.exists():
    raise FileNotFoundError(f"COMBINE_ROOT does not exist: {COMBINE_ROOT}")

for network_dir in sorted(COMBINE_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    net = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        st = station_dir.name

        # Look for combined insitu+ERA5 file(s)
        csv_list = list(station_dir.glob("*_insitu_era5_soil_temperature_level1.csv"))
        if len(csv_list) == 0:
            missing_file.append((net, st))
            continue

        csv_path = csv_list[0]

        try:
            df = pd.read_csv(csv_path)
        except Exception:
            missing_file.append((net, st))
            continue

        # Check required columns
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            missing_cols.append((net, st, missing))
            continue

        # Check for any data rows
        if df.empty:
            empty_overlap.append((net, st))
            continue

        # ------------------------------------------------------------------
        # Validity checks for ts_station_k and ts_station
        # ------------------------------------------------------------------
        if df["ts_station_k"].isna().all():
            all_nan_ts_station_k.append((net, st))

        if df["ts_station"].isna().all():
            all_nan_ts_station.append((net, st))

        # Check whether there is at least one timestamp
        # with both values present (valid pair)
        both_ok = (~df["ts_station_k"].isna()) & (~df["ts_station"].isna())
        if not both_ok.any():
            no_valid_pairs.append((net, st))

# ------------------------------------------------------------------
# Print summary
# ------------------------------------------------------------------
print("\n=== Stations with NO combined file ===")
if not missing_file:
    print("  None")
else:
    for net, st in sorted(missing_file):
        print(f"  {net} / {st}")

print("\n=== Stations with combined file but MISSING columns ===")
if not missing_cols:
    print("  None")
else:
    for net, st, miss in missing_cols:
        print(f"  {net} / {st} -> missing: {', '.join(miss)}")

print("\n=== Stations with empty combined file (no rows) ===")
if not empty_overlap:
    print("  None")
else:
    for net, st in empty_overlap:
        print(f"  {net} / {st}")

print("\n=== Stations with INVALID ts_station_k (all NaN) ===")
if not all_nan_ts_station_k:
    print("  None")
else:
    for net, st in all_nan_ts_station_k:
        print(f"  {net} / {st}")

print("\n=== Stations with INVALID ts_station (ERA5, all NaN) ===")
if not all_nan_ts_station:
    print("  None")
else:
    for net, st in all_nan_ts_station:
        print(f"  {net} / {st}")

print("\n=== Stations with NO valid ts_station_k–ts_station pairs ===")
if not no_valid_pairs:
    print("  None")
else:
    for net, st in no_valid_pairs:
        print(f"  {net} / {st}")



=== Stations with NO combined file ===
  None

=== Stations with combined file but MISSING columns ===
  None

=== Stations with empty combined file (no rows) ===
  None

=== Stations with INVALID ts_station_k (all NaN) ===
  None

=== Stations with INVALID ts_station (ERA5, all NaN) ===
  None

=== Stations with NO valid ts_station_k–ts_station pairs ===
  None


In [6]:
from pathlib import Path
import pandas as pd

# Root where all combined ERA5–in situ files live
ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine")

n_files_scanned = 0
n_files_renamed = 0

for network_dir in sorted(ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    net = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        st = station_dir.name

        # Rename in every CSV in this station directory
        for csv_path in station_dir.glob("*.csv"):
            n_files_scanned += 1
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f"[WARN] Could not read {csv_path}: {e}")
                continue

            if "ts_station" not in df.columns:
                # nothing to change here
                continue

            # Avoid accidental overwrite if ts_era5 already exists
            if "ts_era5" in df.columns:
                print(f"[WARN] {csv_path} already has 'ts_era5', skipping.")
                continue

            df = df.rename(columns={"ts_station": "ts_era5"})
            df.to_csv(csv_path, index=False, float_format="%.3f")
            n_files_renamed += 1
            print(f"Renamed ts_station -> ts_era5 in {csv_path}")

print(f"\nTotal CSV files scanned: {n_files_scanned}")
print(f"Files updated (renamed): {n_files_renamed}")


Renamed ts_station -> ts_era5 in /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Anthony/Anthony_insitu_era5_soil_temperature_level1.csv
Renamed ts_station -> ts_era5 in /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Ashton/Ashton_insitu_era5_soil_temperature_level1.csv
Renamed ts_station -> ts_era5 in /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Byron/Byron_insitu_era5_soil_temperature_level1.csv
Renamed ts_station -> ts_era5 in /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Lamont-CF1/Lamont-CF1_insitu_era5_soil_temperature_level1.csv
Renamed ts_station -> ts_era5 in /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/MapleCity/MapleCity_insitu_era5_soil_temperature_level1.csv
Renamed ts_station -> ts_era5 in /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM/Marshall/Marshall_insitu_era5_soil_temperature_lev

## Checking standard deviation 

In [8]:
import pandas as pd
from pathlib import Path

# Root dirs
ERA5_COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine")
OUT_SUMMARY = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/era5_insitu_sd_corr_summary.csv")

results = []

for network_dir in sorted(ERA5_COMBINE_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    network = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        station = station_dir.name

        csv_files = list(station_dir.glob("*.csv"))
        if not csv_files:
            print(f"[WARN] No CSV in {station_dir}, skipping.")
            continue

        csv_path = csv_files[0]

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"[ERROR] Cannot read {csv_path}: {e}")
            results.append({
                "network": network,
                "station": station,
                "n_points": 0,
                "sd_insitu": None,
                "sd_era5": None,
                "sd_ratio_era5_insitu": None,
                "corr_era5_insitu": None,
                "pass_sd_filter": False,
                "pass_corr_filter": False,
            })
            continue

        if "date" in df.columns and "time" in df.columns:
            df["datetime"] = pd.to_datetime(
                df["date"].astype(str) + " " + df["time"].astype(str),
                errors="coerce"
            )

        if not {"ts_station_k", "ts_era5"}.issubset(df.columns):
            print(f"[WARN] Missing ts_station_k or ts_era5 in {csv_path}, skipping station.")
            continue

        sub = df[["ts_station_k", "ts_era5"]].dropna()
        n_points = len(sub)

        if n_points < 2:
            print(f"[WARN] Not enough valid points for {network}/{station} ({n_points}), skipping stats.")
            sd_insitu = sd_era5 = ratio = corr = None
            pass_sd = pass_corr = False
        else:
            sd_insitu = sub["ts_station_k"].std()
            sd_era5 = sub["ts_era5"].std()
            ratio = sd_era5 / sd_insitu if sd_insitu and sd_insitu > 0 else None
            corr = sub["ts_station_k"].corr(sub["ts_era5"])

            if ratio is not None and 0.1 <= ratio <= 3:
                pass_sd = True
            else:
                pass_sd = False

            pass_corr = corr is not None and corr >= 0.5

        results.append({
            "network": network,
            "station": station,
            "n_points": n_points,
            "sd_insitu": sd_insitu,
            "sd_era5": sd_era5,
            "sd_ratio_era5_insitu": ratio,
            "corr_era5_insitu": corr,
            "pass_sd_filter": pass_sd,
            "pass_corr_filter": pass_corr,
        })

# Build DataFrame
summary_df = pd.DataFrame(results)

# Round numeric columns to 3 decimals
num_cols = ["sd_insitu", "sd_era5", "sd_ratio_era5_insitu", "corr_era5_insitu"]
for col in num_cols:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].round(3)

# Save summary (without csv_path)
summary_df.to_csv(OUT_SUMMARY, index=False)
print(f"Saved summary to {OUT_SUMMARY}")


Saved summary to /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/era5_insitu_sd_corr_summary.csv


## Checking the summary of the SD file

In [9]:
import pandas as pd
from pathlib import Path

summary_path = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/era5_insitu_sd_corr_summary.csv")

df = pd.read_csv(summary_path)

# Total stations
total = len(df)

# Stations that pass both SD and correlation filters
pass_both = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]

# Stations failing at least one filter
fail_any = df[(df["pass_sd_filter"] == False) | (df["pass_corr_filter"] == False)]

print(f"Total stations          : {total}")
print(f"Pass both (SD & corr)   : {len(pass_both)}")
print(f"Fail SD or corr (or both): {len(fail_any)}")

# Optional: counts per network
print("\nPer network (pass both):")
print(pass_both.groupby("network")["station"].count())

print("\nPer network (fail any):")
print(fail_any.groupby("network")["station"].count())

# NEW: print failing station names per network
print("\nFailing stations by network:")
for network, sub in fail_any.groupby("network"):
    station_list = sorted(sub["station"].unique())
    print(f"{network}: {station_list} = {len(station_list)}")


Total stations          : 1362
Pass both (SD & corr)   : 1331
Fail SD or corr (or both): 31

Per network (pass both):
network
ARM                    16
BIEBRZA_S-1            18
BNZ-LTER                9
Berlin                  1
COSMOS-UK              49
CTP_SMTMN              57
CW3E                    8
DAHRA                   1
FLUXNET-AMERIFLUX       7
FMI                    25
FR_Aqui                 5
HOAL                   31
HOBE                   30
HYDROL-NET_PERUGIA      1
IMA_CAN1               12
IPE                     1
LAB-net                 3
LABFLUX                 1
MAQU                   21
MOL-RAO                 2
MySMNet                 3
NAQU                   10
NGARI                  20
OZNET                  17
REMEDHUS               20
RISMA                  23
ROMPS                   5
RSMN                   20
Ru_CFR                  2
SCAN                  204
SKKU                    1
SMN-SDR                33
SMOSMANIA              22
SNOTEL          

## Taking pixle mean

In [ ]:
import os
import glob
import pandas as pd
from collections import Counter
from pathlib import Path


# Root of ERA5–in situ combined files (surface layer, level 1)
COMBINE_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine"

# Output root for pixel-mean processing
OUT_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean"

# Summary file with filters (already created)
SUMMARY_PATH = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/era5_insitu_sd_corr_summary.csv"


def majority(series):
    """Most common non-NaN value in a series."""
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return pd.NA
    return Counter(vals).most_common(1)[0][0]


def classify_elevation(elev):
    """Classify elevation using DEM-I/II/III/IV thresholds."""
    if pd.isna(elev):
        return "DEM-Unknown"
    try:
        elevation = float(elev)
    except Exception:
        return "DEM-Unknown"

    if elevation < 635.523:
        return "DEM-I"
    elif elevation < 1237.128:
        return "DEM-II"
    elif elevation < 1838.733:
        return "DEM-III"
    else:
        return "DEM-IV"


def load_pass_stations(summary_path):
    """
    Read summary CSV and return a set of (network, station) pairs
    that passed BOTH SD and correlation filters.
    """
    df = pd.read_csv(summary_path)
    ok = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]
    return set(zip(ok["network"], ok["station"]))


def process_network(network, pass_stations):
    base_in = os.path.join(COMBINE_ROOT, network)
    base_out = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)  # only base

    print(f"\n=== Network: {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    all_records = []
    used_station_dirs = []

    # Load station data, but only for stations that passed filters
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        # Skip station if it failed any filter in summary
        if (network, station_name) not in pass_stations:
            print(f"  Skipping {network}/{station_name} (failed filter in summary).")
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            df = pd.read_csv(f)

            # Required columns for surface layer, level 1
            required_cols = {"date", "time", "ts_station_k", "ts_era5", "lat_era5", "lon_era5"}
            if not required_cols.issubset(df.columns):
                print(f"  [WARN] Missing required columns in {f}, skipping this file.")
                continue

            df["station"] = station_name
            all_records.append(df)
            used_station_dirs.append(station_dir)

    if not all_records:
        print(f"No valid CSV files found under {base_in}/* (after filtering), skipping.")
        return

    df_all = pd.concat(all_records, ignore_index=True)
    print(f"Loaded {len(set(used_station_dirs))} stations after filtering, total rows: {len(df_all)}")

    # DENSE PIXELS: at least 2 stations in same (lat_era5, lon_era5)
    pixel_station_counts = (
        df_all.groupby(["lat_era5", "lon_era5"])["station"]
              .nunique()
              .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_era5", "lon_era5"]]
    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    n_sparse_files = 0
    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # DENSE
    print("Processing dense pixels (pixel-mean files)...")
    df_dense_all = df_all.merge(dense_pixels, on=["lat_era5", "lon_era5"], how="inner")
    print("Rows in dense pixels (before grouping):", len(df_dense_all))

    stations_in_dense = set()

    # Also read existing dense files to populate stations_in_dense (idempotent runs)
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    if not df_dense_all.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_era5"]
            plon = pix["lon_era5"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            # Skip this pixel if its dense file already exists
            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_era5={plat}, lon_era5={plon} (file exists).")
                continue

            df_pix = df_dense_all[
                (df_dense_all["lat_era5"] == plat) &
                (df_dense_all["lon_era5"] == plon)
            ].copy()

            if df_pix.empty:
                continue

            print(f"  Dense pixel at lat_era5={plat}, lon_era5={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                # Only consider rows with valid in-situ value
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    # pixel-mean in-situ reference
                    ts_ref = g_valid["ts_station_k"].mean()
                    # ERA5 ts_era5 is identical across stations for a pixel/time
                    ts_era5 = g_valid["ts_era5"].iloc[0]

                    lat_val = majority(g_valid["lat"])
                    lon_val = majority(g_valid["lon"])
                    cc_val = majority(g_valid["cc"])
                    lc_val = majority(g_valid["lc"])
                    lcg_val = majority(g_valid["land_cover_group"])
                    clim_val = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    # elevation: mean of station elevations
                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref = pd.NA
                    ts_era5 = pd.NA
                    lat_val = pd.NA
                    lon_val = pd.NA
                    cc_val = pd.NA
                    lc_val = pd.NA
                    lcg_val = pd.NA
                    clim_val = pd.NA
                    tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,   # pixel-mean in-situ
                    "ts_era5": ts_era5,       # ERA5 stl1 at that pixel
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_era5"] = pd.to_numeric(df_pix_mean["ts_era5"], errors="coerce").round(3)

            # retain pixel coordinates and station list
            df_pix_mean["lat_era5"] = plat
            df_pix_mean["lon_era5"] = plon
            df_pix_mean["stations"] = stations_str

            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")
    else:
        print("No dense pixels found to process.")

    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")

    # SPARSE: station-wise files, excluding stations used in any dense pixel
    print("Writing sparse (station-wise) files (excluding dense stations)...")
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        # Skip if station failed filters
        if (network, station_name) not in pass_stations:
            continue

        # Skip if this station was already included in any dense pixel
        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            # Skip if sparse file already exists
            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            df = pd.read_csv(f)

            # Rename in-situ column to match dense output naming
            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            # ERA5 already named ts_era5; keep as is
            df = df.rename(columns=rename_map)

            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Stations found (raw): {n_stations_found}")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")


if __name__ == "__main__":
    # Load list of stations that passed ERA5 SD & correlation filters
    pass_stations = load_pass_stations(SUMMARY_PATH)

    # Discover networks under COMBINE_ROOT
    network_dirs = sorted(
        d for d in glob.glob(os.path.join(COMBINE_ROOT, "*"))
        if os.path.isdir(d)
    )
    networks = [os.path.basename(d) for d in network_dirs]

    print("Networks found:", networks)
    for net in networks:
        process_network(net, pass_stations)


Networks found: ['ARM', 'BIEBRZA_S-1', 'BNZ-LTER', 'Berlin', 'COSMOS-UK', 'CTP_SMTMN', 'CW3E', 'DAHRA', 'FLUXNET-AMERIFLUX', 'FMI', 'FR_Aqui', 'HOAL', 'HOBE', 'HYDROL-NET_PERUGIA', 'IMA_CAN1', 'IPE', 'LAB-net', 'LABFLUX', 'MAQU', 'MOL-RAO', 'MySMNet', 'NAQU', 'NGARI', 'OZNET', 'REMEDHUS', 'RISMA', 'ROMPS', 'RSMN', 'Ru_CFR', 'SCAN', 'SKKU', 'SMN-SDR', 'SMOSMANIA', 'SNOTEL', 'SOILSCAPE', 'STEMS', 'SWEX_POLAND', 'TAHMO', 'TERENO', 'TWENTE', 'TxSON', 'USCRN', 'WSMN', 'XMS-CAT']

=== Network: ARM ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/ARM
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/ARM
Found 16 station directories.
Loaded 16 stations after filtering, total rows: 992395
Total unique pixels: 16
Dense pixels (n_stations >= 2): 0
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 0
No dense pixels found to process.
Stations that appear in dense pixels: 0
Writing sparse (sta

/tmp/ipykernel_56736/3663253199.py:135: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


Loaded 57 stations after filtering, total rows: 2254944
Total unique pixels: 13
Dense pixels (n_stations >= 2): 12
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 2213576


/tmp/ipykernel_56736/3663253199.py:135: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)
/tmp/ipykernel_56736/3663253199.py:135: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_era5=31.0, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.0, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=92.0 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.5, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.5, lon_era5=92.0 (file exists).
  Skipping dense pixel lat_era5=31.5, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.75, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.75, lon_era5=92.25 (file exists).
  Skipping dense pixel lat_era5=31.75, lon_era5=92.5 (file exists).
  Skipping dense pixel lat_era5=32.0, lon_era5=91.75 (file exists).
Stations that appear in dense pixels: 56
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for CTP_SMTMN/M15 (/home/aminr/Desktop/Master_Thesi

...

  Skipping dense pixel lat_era5=2.0, lon_era5=103.5 (file exists).
Stations that appear in dense pixels: 3
Writing sparse (station-wise) files (excluding dense stations)...
Finished sparse files. Processed 0 station CSV files (excluding 3 dense stations).
All done for MySMNet.
===== SUMMARY =====
Stations found (raw): 3
Dense pixels processed (with valid data): 0
Dense pixel files written: 0

=== Network: NAQU ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine/NAQU
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/NAQU
Found 10 station directories.
Loaded 10 stations after filtering, total rows: 424918
Total unique pixels: 3
Dense pixels (n_stations >= 2): 2
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 357454
  Skipping dense pixel lat_era5=31.25, lon_era5=91.75 (file exists).
  Skipping dense pixel lat_era5=31.25, lon_era5=92.0 (file exists).
Stations that appear in dense pix

/tmp/ipykernel_56736/3663253199.py:135: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


Loaded 20 stations after filtering, total rows: 947142
Total unique pixels: 6
Dense pixels (n_stations >= 2): 4
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 852809
  Skipping dense pixel lat_era5=32.5, lon_era5=79.75 (file exists).
  Skipping dense pixel lat_era5=32.5, lon_era5=80.0 (file exists).
  Skipping dense pixel lat_era5=32.75, lon_era5=80.0 (file exists).
  Skipping dense pixel lat_era5=33.5, lon_era5=79.75 (file exists).
Stations that appear in dense pixels: 18
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for NGARI/SQ14 (/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/NGARI/sparse/SQ14_insitu_era5_soil_temperature_level1.csv exists).
  Skipping sparse file for NGARI/SQ21 (/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/NGARI/sparse/SQ21_insitu_era5_soil_temperature_level1.csv exists).
Finished sparse files. Processed 0 station CSV file

/tmp/ipykernel_56736/3663253199.py:135: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_era5=20.0, lon_era5=-155.5 (file exists).
  Skipping dense pixel lat_era5=35.0, lon_era5=-119.5 (file exists).
  Skipping dense pixel lat_era5=35.0, lon_era5=-86.5 (file exists).
  Skipping dense pixel lat_era5=36.25, lon_era5=-115.75 (file exists).
  Skipping dense pixel lat_era5=36.25, lon_era5=-115.5 (file exists).
  Skipping dense pixel lat_era5=37.75, lon_era5=-109.25 (file exists).
  Skipping dense pixel lat_era5=38.5, lon_era5=-92.25 (file exists).
  Skipping dense pixel lat_era5=40.75, lon_era5=-104.75 (file exists).
  Skipping dense pixel lat_era5=41.25, lon_era5=-111.25 (file exists).
Stations that appear in dense pixels: 20
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for SCAN/AAMU-JTG (/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/SCAN/sparse/AAMU-JTG_insitu_era5_soil_temperature_level1.csv exists).
  Skipping sparse file for SCAN/Abrams (/home/aminr/Desktop/Master_Thesis/G

## Validation of pixel_mean files

### 1. check all networks/station/files

In [2]:
import pandas as pd
from pathlib import Path
import glob
import os

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine")
OUT_ROOT     = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean")
SUMMARY_PATH = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/era5_insitu_sd_corr_summary.csv")

summary = pd.read_csv(SUMMARY_PATH)

# Stations that passed both filters
passed = summary[(summary["pass_sd_filter"] == True) & (summary["pass_corr_filter"] == True)]
passed_set = set(zip(passed["network"], passed["station"]))

# Stations that failed any filter
failed = summary[(summary["pass_sd_filter"] == False) | (summary["pass_corr_filter"] == False)]
failed_set = set(zip(failed["network"], failed["station"]))

dense_stations = set()
sparse_stations = set()

for net_dir in sorted(COMBINE_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    net_out = OUT_ROOT / network
    dense_dir = net_out / "dense"
    sparse_dir = net_out / "sparse"

    # collect from dense
    if dense_dir.exists():
        for f in glob.glob(str(dense_dir / f"{network}_dense_lat*_lon*.csv")):
            df = pd.read_csv(f)
            if "stations" in df.columns and not df.empty:
                for st in str(df["stations"].iloc[0]).split(","):
                    st = st.strip()
                    if st:
                        dense_stations.add((network, st))

    # collect from sparse
    if sparse_dir.exists():
        for st_dir in sorted((COMBINE_ROOT / network).iterdir()):
            if not st_dir.is_dir():
                continue
            station = st_dir.name
            pattern = str(sparse_dir / f"{station}*.csv")
            files = glob.glob(pattern)
            if files:
                sparse_stations.add((network, station))

print("Total passed stations        :", len(passed_set))
print("Stations in dense pixels     :", len(dense_stations))
print("Stations in sparse outputs   :", len(sparse_stations))

print("\nStations that passed but not in dense or sparse:")
missing = passed_set - dense_stations - sparse_stations
print(missing)

print("\nStations that failed but appear in dense or sparse (should be empty):")
bad = (failed_set & dense_stations) | (failed_set & sparse_stations)
print(bad)


/tmp/ipykernel_58173/1736780726.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/1736780726.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/1736780726.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/1736780726.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/1736780726.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/1736780726.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read

Total passed stations        : 1331
Stations in dense pixels     : 634
Stations in sparse outputs   : 700

Stations that passed but not in dense or sparse:
set()

Stations that failed but appear in dense or sparse (should be empty):
{('SNOTEL', 'BakerButte')}


### 2. checking dense files: ts_reference and metadata

In [4]:
import pandas as pd
from pathlib import Path
import glob
import numpy as np

NETWORK = "BIEBRZA_S-1"  # change to other networks as needed

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine") / NETWORK
DENSE_DIR    = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean") / NETWORK / "dense"

def majority(series):
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return pd.NA
    return max(set(vals), key=vals.count)

def classify_elevation(elev):
    if pd.isna(elev):
        return "DEM-Unknown"
    try:
        elevation = float(elev)
    except Exception:
        return "DEM-Unknown"
    if elevation < 635.523:
        return "DEM-I"
    elif elevation < 1237.128:
        return "DEM-II"
    elif elevation < 1838.733:
        return "DEM-III"
    else:
        return "DEM-IV"

dense_files = glob.glob(str(DENSE_DIR / f"{NETWORK}_dense_lat*_lon*.csv"))
print(f"Found {len(dense_files)} dense files for {NETWORK}")

mismatches = []

for f_dense in dense_files:
    df_dense = pd.read_csv(f_dense)
    if df_dense.empty:
        continue

    stations = str(df_dense["stations"].iloc[0]).split(",")
    stations = [s.strip() for s in stations if s.strip()]

    # load original station data for these stations
    all_records = []
    for st in stations:
        st_dir = COMBINE_ROOT / st
        for f in st_dir.glob("*.csv"):
            df = pd.read_csv(f)
            df["station"] = st
            all_records.append(df)

    if not all_records:
        continue

    df_all = pd.concat(all_records, ignore_index=True)

    # pixel coordinate from dense file
    plat = df_dense["lat_era5"].iloc[0]
    plon = df_dense["lon_era5"].iloc[0]
    df_pix = df_all[(df_all["lat_era5"] == plat) & (df_all["lon_era5"] == plon)].copy()

    # merge on date+time to compare row-by-row
    df_pix["key"] = df_pix["date"].astype(str) + " " + df_pix["time"].astype(str)
    df_dense["key"] = df_dense["date"].astype(str) + " " + df_dense["time"].astype(str)

    for key, g in df_pix.groupby("key"):
        dense_row = df_dense[df_dense["key"] == key]
        if dense_row.empty:
            continue

        dense_row = dense_row.iloc[0]

        g_valid = g[g["ts_station_k"].notna()]
        n_valid = g_valid["station"].nunique()

        # recompute reference and metadata
        if n_valid >= 2:
            ts_ref_true = round(g_valid["ts_station_k"].mean(), 3)
            ts_era5_true = g_valid["ts_era5"].iloc[0]

            lat_true  = majority(g_valid["lat"])
            lon_true  = majority(g_valid["lon"])
            cc_true   = majority(g_valid["cc"])
            lc_true   = majority(g_valid["lc"])
            lcg_true  = majority(g_valid["land_cover_group"])
            clim_true = majority(g_valid["climate_group"])
            tcls_true = majority(g_valid["temp_class"])
            elev_mean_true = g_valid["elev"].mean()
            elevc_true = classify_elevation(elev_mean_true)
        else:
            ts_ref_true = np.nan
            ts_era5_true = np.nan
            lat_true = lon_true = cc_true = lc_true = lcg_true = clim_true = tcls_true = np.nan
            elev_mean_true = np.nan
            elevc_true = "DEM-Unknown"

        # compare with dense file
        def neq(a, b, tol=1e-3):
            if pd.isna(a) and pd.isna(b):
                return False
            if isinstance(a, (float, int)) and isinstance(b, (float, int)):
                return abs(a - b) > tol
            return a != b

        issues = []

        if neq(dense_row["ts_reference"], ts_ref_true):
            issues.append(f"ts_reference {dense_row['ts_reference']} vs {ts_ref_true}")
        if neq(dense_row["ts_era5"], ts_era5_true):
            issues.append(f"ts_era5 {dense_row['ts_era5']} vs {ts_era5_true}")
        if neq(dense_row["lat"], lat_true):
            issues.append(f"lat {dense_row['lat']} vs {lat_true}")
        if neq(dense_row["lon"], lon_true):
            issues.append(f"lon {dense_row['lon']} vs {lon_true}")
        if neq(dense_row["cc"], cc_true):
            issues.append(f"cc {dense_row['cc']} vs {cc_true}")
        if neq(dense_row["lc"], lc_true):
            issues.append(f"lc {dense_row['lc']} vs {lc_true}")
        if neq(dense_row["land_cover_group"], lcg_true):
            issues.append(f"land_cover_group {dense_row['land_cover_group']} vs {lcg_true}")
        if neq(dense_row["climate_group"], clim_true):
            issues.append(f"climate_group {dense_row['climate_group']} vs {clim_true}")
        if neq(dense_row["temp_class"], tcls_true):
            issues.append(f"temp_class {dense_row['temp_class']} vs {tcls_true}")
        if neq(dense_row["elev"], elev_mean_true):
            issues.append(f"elev {dense_row['elev']} vs {elev_mean_true}")
        if neq(dense_row["elevation_class"], elevc_true):
            issues.append(f"elevation_class {dense_row['elevation_class']} vs {elevc_true}")

        if issues:
            mismatches.append((f_dense, key, issues))

print(f"\nTotal mismatches found: {len(mismatches)}")
for f_dense, key, issues in mismatches[:20]:
    print(f"Dense file: {f_dense}, datetime: {key}")
    for issue in issues:
        print("  -", issue)


Found 2 dense files for BIEBRZA_S-1

Total mismatches found: 57726
Dense file: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/BIEBRZA_S-1/dense/BIEBRZA_S-1_dense_lat53.5_lon23.0.csv, datetime: 2015-10-03 16:00
  - lat 53.604 vs 53.606
Dense file: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/BIEBRZA_S-1/dense/BIEBRZA_S-1_dense_lat53.5_lon23.0.csv, datetime: 2015-10-03 17:00
  - lat 53.604 vs 53.606
Dense file: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/BIEBRZA_S-1/dense/BIEBRZA_S-1_dense_lat53.5_lon23.0.csv, datetime: 2015-10-03 18:00
  - lat 53.604 vs 53.606
Dense file: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/BIEBRZA_S-1/dense/BIEBRZA_S-1_dense_lat53.5_lon23.0.csv, datetime: 2015-10-03 19:00
  - lat 53.604 vs 53.606
Dense file: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/BIEBRZA_S-1/dense/BIEBRZA_S-1_dense_lat53.5_lon23.0.

### 3. Check sparse files: full copy correctness

In [5]:
import pandas as pd
from pathlib import Path
import glob

NETWORK = "SNOTEL"  # change as needed

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine") / NETWORK
SPARSE_DIR   = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean") / NETWORK / "sparse"

# Collect all sparse files
sparse_files = glob.glob(str(SPARSE_DIR / "*.csv"))
print(f"Found {len(sparse_files)} sparse files for {NETWORK}")

problems = []

for f_sparse in sparse_files:
    df_s = pd.read_csv(f_sparse)
    station_name = Path(f_sparse).stem.split("_")[0]  # adapt if station naming differs

    # find original combine file: assume same filename under COMBINE_ROOT/<station>/
    # if your naming differs, adapt this.
    st_dir = COMBINE_ROOT / station_name
    orig_candidates = list(st_dir.glob(Path(f_sparse).name))
    if not orig_candidates:
        problems.append((f_sparse, "Original file not found"))
        continue

    df_o = pd.read_csv(orig_candidates[0])

    # apply same renaming as script did
    rename_map = {}
    if "ts_station_k" in df_o.columns:
        rename_map["ts_station_k"] = "ts_reference"
    if "ts_era5" in df_o.columns:
        rename_map["ts_era5"] = "ts_era5"
    df_o = df_o.rename(columns=rename_map)

    # Compare shapes
    if df_o.shape != df_s.shape:
        problems.append((f_sparse, f"Shape mismatch: orig {df_o.shape}, sparse {df_s.shape}"))
        continue

    # Compare values cell by cell
    if not df_o.equals(df_s):
        problems.append((f_sparse, "Data mismatch between original and sparse copy"))

print(f"\nSparse problems found: {len(problems)}")
for p in problems[:20]:
    print(p)


Found 239 sparse files for SNOTEL

Sparse problems found: 0


### 4. Check for files without any valid reference data

In [6]:
from pathlib import Path
import pandas as pd
import glob
import os

OUT_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean")

dense_empty = []
sparse_empty = []

for net_dir in sorted(OUT_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    dense_dir = net_dir / "dense"
    sparse_dir = net_dir / "sparse"

    # Dense
    if dense_dir.exists():
        for f in glob.glob(str(dense_dir / "*.csv")):
            df = pd.read_csv(f)
            if "ts_reference" in df.columns:
                if df["ts_reference"].notna().sum() == 0:
                    dense_empty.append(f)

    # Sparse
    if sparse_dir.exists():
        for f in glob.glob(str(sparse_dir / "*.csv")):
            df = pd.read_csv(f)
            if "ts_reference" in df.columns:
                if df["ts_reference"].notna().sum() == 0:
                    sparse_empty.append(f)

print("Dense files with no valid ts_reference:", len(dense_empty))
for f in dense_empty[:20]:
    print("  ", f)

print("\nSparse files with no valid ts_reference:", len(sparse_empty))
for f in sparse_empty[:20]:
    print("  ", f)


/tmp/ipykernel_58173/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_58173/599062092.py:21: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f

Dense files with no valid ts_reference: 1
   /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/era5/pixel_mean/TWENTE/dense/TWENTE_dense_lat52.5_lon7.0.csv

Sparse files with no valid ts_reference: 0


In [7]:
import os
import glob
from pathlib import Path
import pandas as pd

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/combine")
OUT_ROOT     = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean")
SUMMARY_PATH = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/era5_insitu_sd_corr_summary.csv")

# ------------------------------------------------------------------
# 1) Load summary to know which stations passed/failed
# ------------------------------------------------------------------
summary = pd.read_csv(SUMMARY_PATH)

passed = summary[(summary["pass_sd_filter"] == True) & (summary["pass_corr_filter"] == True)]
passed_set = set(zip(passed["network"], passed["station"]))

failed = summary[(summary["pass_sd_filter"] == False) | (summary["pass_corr_filter"] == False)]
failed_set = set(zip(failed["network"], failed["station"]))

# ------------------------------------------------------------------
# 2) Scan COMBINE_ROOT: all networks / stations / files
# ------------------------------------------------------------------
combine_records = []

for net_dir in sorted(COMBINE_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    for st_dir in sorted(net_dir.iterdir()):
        if not st_dir.is_dir():
            continue
        station = st_dir.name
        files = sorted(glob.glob(str(st_dir / "*.csv")))
        for f in files:
            combine_records.append({
                "network": network,
                "station": station,
                "file_basename": os.path.basename(f),
                "source": "combine",
            })

combine_df = pd.DataFrame(combine_records)

print("COMBINE: networks =", combine_df["network"].nunique(),
      "stations =", combine_df[["network","station"]].drop_duplicates().shape[0],
      "files =", len(combine_df))

# ------------------------------------------------------------------
# 3) Scan OUT_ROOT: dense and sparse outputs, map back to stations
# ------------------------------------------------------------------
output_records = []

for net_dir in sorted(OUT_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name

    dense_dir = net_dir / "dense"
    if dense_dir.is_dir():
        for f in glob.glob(str(dense_dir / f"{network}_dense_lat*_lon*.csv")):
            try:
                df = pd.read_csv(f, usecols=["stations"])
                if df.empty:
                    stations = []
                else:
                    stations = [s.strip() for s in str(df["stations"].iloc[0]).split(",") if s.strip()]
            except Exception:
                stations = []

            if not stations:
                # still record file with unknown stations
                output_records.append({
                    "network": network,
                    "station": None,
                    "file_basename": os.path.basename(f),
                    "type": "dense",
                })
            else:
                for st in stations:
                    output_records.append({
                        "network": network,
                        "station": st,
                        "file_basename": os.path.basename(f),
                        "type": "dense",
                    })

    sparse_dir = net_dir / "sparse"
    if sparse_dir.is_dir():
        for f in glob.glob(str(sparse_dir / "*.csv")):
            # sparse file name is typically <station>*.csv
            base = os.path.basename(f)
            station = base.split("_")[0]  # adjust if naming different
            output_records.append({
                "network": network,
                "station": station,
                "file_basename": base,
                "type": "sparse",
            })

output_df = pd.DataFrame(output_records)

print("PIXEL_MEAN: networks =", output_df["network"].nunique(),
      "stations =", output_df[["network","station"]].dropna().drop_duplicates().shape[0],
      "files =", len(output_df))

# ------------------------------------------------------------------
# 4) Compare at station level
# ------------------------------------------------------------------
combine_stations = set(zip(combine_df["network"], combine_df["station"]))
output_stations  = set(zip(output_df["network"], output_df["station"]))

stations_missing_any_output = combine_stations - output_stations
stations_with_output_but_no_input = output_stations - combine_stations

print("\nStations present in COMBINE but missing in any pixel_mean output:", len(stations_missing_any_output))
print(sorted(stations_missing_any_output)[:50])  # show first 50

print("\nStations present in pixel_mean but not in COMBINE:", len(stations_with_output_but_no_input))
print(sorted(stations_with_output_but_no_input)[:50])

# ------------------------------------------------------------------
# 5) Compare at file level per station
#     - For passed stations, every input file should correspond
#       to at least one dense or sparse output file.
# ------------------------------------------------------------------
# Build quick lookup: for each (net, station) -> list of output files
out_by_station = (
    output_df
    .groupby(["network","station"])["file_basename"]
    .apply(set)
    .to_dict()
)

file_mismatches = []

for (net, st), group in combine_df.groupby(["network","station"]):
    input_files = set(group["file_basename"])
    out_files = out_by_station.get((net, st), set())

    # Only enforce matching for stations that passed filters
    status = "passed" if (net, st) in passed_set else "failed_or_unknown"

    if status == "passed":
        if not out_files:
            file_mismatches.append({
                "network": net,
                "station": st,
                "status": status,
                "missing_all_outputs": True,
                "missing_files": ";".join(sorted(input_files)),
            })
        # you could also go finer: per file mapping if there is a 1:1 naming
    else:
        # For failed stations, we might want to know if they still produced outputs
        if out_files:
            file_mismatches.append({
                "network": net,
                "station": st,
                "status": status,
                "missing_all_outputs": False,
                "missing_files": "",
            })

mismatch_df = pd.DataFrame(file_mismatches)
print("\nPassed stations with no outputs OR failed stations with outputs:", len(mismatch_df))
print(mismatch_df.head(50))

# Save detailed reports
combine_df.to_csv(OUT_ROOT / "audit_combine_listing.csv", index=False)
output_df.to_csv(OUT_ROOT / "audit_pixel_mean_listing.csv", index=False)
mismatch_df.to_csv(OUT_ROOT / "audit_mismatches.csv", index=False)


COMBINE: networks = 44 stations = 1362 files = 1362
PIXEL_MEAN: networks = 43 stations = 1331 files = 1331

Stations present in COMBINE but missing in any pixel_mean output: 31
[('BNZ-LTER', 'FP3A'), ('FMI', 'SOD022'), ('FMI', 'SOD023'), ('HOAL', 'Hoal-19'), ('OZNET', 'Alabama'), ('OZNET', 'Dry-Lake'), ('SCAN', 'Combate'), ('SCAN', 'Corozal'), ('SCAN', 'MaricaoForest'), ('SCAN', 'MooseInc'), ('SCAN', 'WaimeaPlain'), ('SNOTEL', 'BakerButte'), ('SNOTEL', 'Coldfoot'), ('SNOTEL', 'ElkButte'), ('SNOTEL', 'FortValley'), ('SNOTEL', 'JackWadeJct'), ('SNOTEL', 'LostDog'), ('SNOTEL', 'MormonMtnSummit'), ('SNOTEL', 'TogwoteePass'), ('SOILSCAPE', 'node401'), ('TAHMO', 'CoteDIvoireMeteoOffice'), ('TAHMO', 'DedanKimathiUniversityofTechnology'), ('TAHMO', 'ImurtottPrimaryschool'), ('TAHMO', 'MagomanoSecondarySchool'), ('TAHMO', 'MaraTriangleAirstrip'), ('TAHMO', 'MasenoUniversity'), ('TAHMO', 'MurungaruSecondarySchool'), ('TAHMO', 'NaboishoCamp'), ('TAHMO', 'SenchuraSecondarySchool'), ('TAHMO', 'StJo